In [24]:
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression, chi2
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import spearmanr
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
import matplotlib.pyplot as plt
import pandas as pd
import gc

In [25]:
# ==================== CLEAR MEMORY ====================
tf.keras.backend.clear_session()
gc.collect()

0

In [26]:
# ==================== PARAMETERS ====================
SAVE_PATH = "windowed_data"
WINDOW_SIZE = 24
HORIZON = 1
EPOCHS = 16
NUM_BATCHES = 5
RANDOM_STATE = 42

In [27]:
# ==================== 1. LOAD DATA ====================
X_all = []
y_all = []

for b in range(NUM_BATCHES):
    X_batch = np.load(os.path.join(SAVE_PATH, f"X_batch_{b}.npy"))
    y_batch = np.load(os.path.join(SAVE_PATH, f"y_batch_{b}.npy"))
    X_all.append(X_batch)
    y_all.append(y_batch)

X_all = np.concatenate(X_all, axis=0)
y_all = np.concatenate(y_all, axis=0)

print(f"Total samples loaded: {len(X_all)}")
print(f"Shape: X={X_all.shape}, y={y_all.shape}")

Total samples loaded: 164424
Shape: X=(164424, 24), y=(164424, 1)


In [28]:
# ==================== 2. SPLIT DATA: 70% TRAIN, 15% VAL, 15% TEST ====================
n_samples = len(X_all)
train_size = int(0.70 * n_samples)
val_size = int(0.15 * n_samples)

X_train_flat = X_all[:train_size]
y_train = y_all[:train_size].flatten()  # FLATTEN HERE

X_val_flat = X_all[train_size:train_size + val_size]
y_val = y_all[train_size:train_size + val_size].flatten()  # FLATTEN HERE

X_test_flat = X_all[train_size + val_size:]
y_test = y_all[train_size + val_size:].flatten()  # FLATTEN HERE

print(f"Train: {len(X_train_flat)} samples ({len(X_train_flat)/n_samples*100:.1f}%)")
print(f"Val:   {len(X_val_flat)} samples ({len(X_val_flat)/n_samples*100:.1f}%)")
print(f"Test:  {len(X_test_flat)} samples ({len(X_test_flat)/n_samples*100:.1f}%)")

del X_all, y_all
gc.collect()

Train: 115096 samples (70.0%)
Val:   24663 samples (15.0%)
Test:  24665 samples (15.0%)


0

In [29]:
# ==================== 3. FEATURE ENGINEERING FUNCTIONS ====================
# Manual Features
def create_manual_features(X):
    features = []
    features.append(np.mean(X, axis=1))
    features.append(np.std(X, axis=1))
    features.append(np.min(X, axis=1))
    features.append(np.max(X, axis=1))
    features.append(X[:, -1])
    features.append(X[:, -1] - X[:, 0])
    features.append(np.median(X, axis=1))
    features.append(np.percentile(X, 25, axis=1))
    features.append(np.percentile(X, 75, axis=1))
    return np.column_stack(features)
# Fisher Score
def apply_fisher_score(X_train, y_train, X_val, X_test, k=10):
    selector = SelectKBest(score_func=f_regression, k=k)
    X_train_selected = selector.fit_transform(X_train, y_train)
    X_val_selected = selector.transform(X_val)
    X_test_selected = selector.transform(X_test)
    return X_train_selected, X_val_selected, X_test_selected

# mRMR approximation using Mutual Information
def apply_mrmr(X_train, y_train, X_val, X_test, k=10):
    selector = SelectKBest(score_func=mutual_info_regression, k=k)
    X_train_selected = selector.fit_transform(X_train, y_train)
    X_val_selected = selector.transform(X_val)
    X_test_selected = selector.transform(X_test)
    return X_train_selected, X_val_selected, X_test_selected

# PCA dimensionality reduction
def apply_pca(X_train, X_val, X_test, n_components=10):
    pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
    X_train_pca = pca.fit_transform(X_train)
    X_val_pca = pca.transform(X_val)
    X_test_pca = pca.transform(X_test)
    print(f"  PCA variance: {pca.explained_variance_ratio_.sum()*100:.2f}%")
    return X_train_pca, X_val_pca, X_test_pca



In [30]:
# ==================== 4. BUILD & TRAIN LSTM MODEL ====================
def build_and_train_lstm(X_train, y_train, X_val, y_val, model_name, input_dim):
    print(f"\n  Training {model_name}")
    
    # Clear previous model from memory
    tf.keras.backend.clear_session()
    gc.collect()
    
    if X_train.ndim == 2:
        X_train = X_train[..., np.newaxis]
        X_val = X_val[..., np.newaxis]
    
    model = Sequential([
        LSTM(64, input_shape=(X_train.shape[1], 1)),
        Dense(1)
    ])
    
    model.compile(optimizer=Adam(learning_rate=0.001), loss="mse")
    
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)
    
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=64, 
        callbacks=[early_stop],
        verbose=1, 
        shuffle=True
    )
    
    print(f"  ✓ {model_name} trained (epochs: {len(history.history['loss'])})")
    gc.collect()
    
    return model, history

In [31]:
# ==================== 5. EVALUATION FUNCTION ====================
def evaluate_model(model, X_test, y_test, model_name):
    if X_test.ndim == 2:
        X_test = X_test[..., np.newaxis]
    
    y_pred = model.predict(X_test, verbose=0, batch_size=64).reshape(-1)
    
    errors = y_test - y_pred
    mse = np.mean(errors ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(errors))
    
    delta = 15
    accuracy = (np.sum(np.abs(errors) <= delta) / len(errors)) * 100
    
    # MAPE calculation
    mape = np.mean(np.abs(errors) / np.abs(y_test)) * 100
    mape_accuracy = 100 - mape

    
    return {
        'Model': model_name,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'Accuracy (%)': accuracy,
        'MAPE (%)': mape,
        'MAPE Accuracy (%)': mape_accuracy
    }

In [32]:
# ==================== 6. BASELINE MODEL (NAIVE PREDICTION) ====================
y_naive = X_test_flat[:, -1]
errors_naive = y_test - y_naive
mse_naive = np.mean(errors_naive ** 2)
rmse_naive = np.sqrt(mse_naive)
mae_naive = np.mean(np.abs(errors_naive))

delta = 15
correct_count = np.sum(np.abs(errors_naive) <= delta)
total_count = len(errors_naive)
acc_naive = (correct_count / total_count) * 100

# MAPE
mape_naive = np.mean(np.abs(errors_naive) / np.abs(y_test)) * 100
mape_acc_naive = 100 - mape_naive

baseline_results = {
    'Model': 'Baseline (Naive)',
    'MSE': mse_naive,
    'RMSE': rmse_naive,
    'MAE': mae_naive,
    'Accuracy (%)': acc_naive,
    'MAPE (%)': mape_naive,
    'MAPE Accuracy (%)': mape_acc_naive
}

print(f"Baseline RMSE: {rmse_naive:.2f} mg/dL")
print(f"Baseline MAE: {mae_naive:.2f} mg/dL")
print(f"Baseline Accuracy: {acc_naive:.2f}%")
print(f"Baseline MAPE: {mape_naive:.2f}%")
print(f"Baseline MAPE Accuracy: {mape_acc_naive:.2f}%")

del y_naive, errors_naive
gc.collect()

Baseline RMSE: 14.75 mg/dL
Baseline MAE: 9.72 mg/dL
Baseline Accuracy: 80.90%
Baseline MAPE: 6.41%
Baseline MAPE Accuracy: 93.59%


0

In [33]:
# ==================== 7. MODEL 1: RAW FEATURES (NO FEATURE SELECTION) ====================
X_train_raw = X_train_flat.copy()
X_val_raw = X_val_flat.copy()
X_test_raw = X_test_flat.copy()

model_raw, hist_raw = build_and_train_lstm(X_train_raw, y_train, X_val_raw, y_val, 
                                            "Raw Features", X_train_raw.shape[1])
results_raw = evaluate_model(model_raw, X_test_raw, y_test, "Raw Features")

print(f"Raw RMSE: {results_raw['RMSE']:.2f} mg/dL")
print(f"Raw MAE: {results_raw['MAE']:.2f} mg/dL")
print(f"Raw Accuracy: {results_raw['Accuracy (%)']:.2f}%")
print(f"Raw MAPE: {results_raw['MAPE (%)']:.2f}%")
print(f"Raw MAPE Accuracy: {results_raw['MAPE Accuracy (%)']:.2f}%")

del X_train_raw, X_val_raw, X_test_raw, model_raw
gc.collect()


  Training Raw Features
Epoch 1/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 18s 9ms/step - loss: 16846.1582 - val_loss: 5876.5596
Epoch 2/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 4592.9458 - val_loss: 1293.4552
Epoch 3/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - loss: 1476.1395 - val_loss: 435.9102
Epoch 4/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 592.0666 - val_loss: 235.9794
Epoch 5/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 319.3791 - val_loss: 188.5280
Epoch 6/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 212.5099 - val_loss: 165.0560
Epoch 7/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 172.4870 - val_loss: 140.3557
Epoch 8/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 152.8670 - val_loss: 134.3677
Epoch 9/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - loss: 140.9438 - val_loss: 124.4330
Epoch 10/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - loss: 133.5799 - val_loss: 115.8235
Epoch 11/16
1799/1799 ━━━━━━━━━━━━

6643

In [34]:
# ==================== 8. MODEL 2: MANUAL FEATURES ====================
X_train_manual = create_manual_features(X_train_flat)
X_val_manual = create_manual_features(X_val_flat)
X_test_manual = create_manual_features(X_test_flat)


model_manual, hist_manual = build_and_train_lstm(X_train_manual, y_train, X_val_manual, y_val,
                                                  "Manual Features", X_train_manual.shape[1])
results_manual = evaluate_model(model_manual, X_test_manual, y_test, "Manual Features")

print(f"Manual RMSE: {results_manual['RMSE']:.2f} mg/dL")
print(f"Manual MAE: {results_manual['MAE']:.2f} mg/dL")
print(f"Manual Accuracy: {results_manual['Accuracy (%)']:.2f}%")
print(f"Manual MAPE: {results_manual['MAPE (%)']:.2f}%")
print(f"Manual MAPE Accuracy: {results_manual['MAPE Accuracy (%)']:.2f}%")

del X_train_manual, X_val_manual, X_test_manual, model_manual
gc.collect()


  Training Manual Features
Epoch 1/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 18174.0098 - val_loss: 7110.3213
Epoch 2/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 6165.7617 - val_loss: 2089.3818
Epoch 3/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 2149.2385 - val_loss: 736.9597
Epoch 4/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 927.7648 - val_loss: 434.2414
Epoch 5/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 524.5613 - val_loss: 383.4296
Epoch 6/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 410.1416 - val_loss: 330.9114
Epoch 7/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 374.2246 - val_loss: 319.2087
Epoch 8/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 357.3150 - val_loss: 344.4163
Epoch 9/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 352.1598 - val_loss: 324.5708
Epoch 10/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 347.1177 - val_loss: 340.6843
Epoch 11/16
1799/1799 ━━━━━━━━━━━━━

331

In [37]:
# ==================== 9. MODEL 3: FISHER SCORE ====================
K_FISHER = 10
X_train_fisher, X_val_fisher, X_test_fisher = apply_fisher_score(
    X_train_flat, y_train, X_val_flat, X_test_flat, k=K_FISHER
)
print(f"  Fisher: Selected top {K_FISHER} features")

model_fisher, hist_fisher = build_and_train_lstm(X_train_fisher, y_train, X_val_fisher, y_val,
                                                  "Fisher Score", X_train_fisher.shape[1])
results_fisher = evaluate_model(model_fisher, X_test_fisher, y_test, "Fisher Score")

print(f"Fisher RMSE: {results_fisher['RMSE']:.2f} mg/dL")
print(f"Fisher MAE: {results_fisher['MAE']:.2f} mg/dL")
print(f"Fisher Accuracy: {results_fisher['Accuracy (%)']:.2f}%")
print(f"Fisher MAPE: {results_fisher['MAPE (%)']:.2f}%")
print(f"Fisher MAPE Accuracy: {results_fisher['MAPE Accuracy (%)']:.2f}%")

del X_train_fisher, X_val_fisher, X_test_fisher, model_fisher
gc.collect()

  Fisher: Selected top 10 features

  Training Fisher Score
Epoch 1/16


C:\Users\Roaa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1799/1799 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 18463.9961 - val_loss: 7802.2588
Epoch 2/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 6169.4951 - val_loss: 1909.3716
Epoch 3/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 2087.4460 - val_loss: 632.2864
Epoch 4/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 832.0262 - val_loss: 291.6862
Epoch 5/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 381.2529 - val_loss: 170.5643
Epoch 6/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 217.3013 - val_loss: 136.4375
Epoch 7/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 160.2773 - val_loss: 120.0445
Epoch 8/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 140.1895 - val_loss: 119.8330
Epoch 9/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 132.6148 - val_loss: 113.4868
Epoch 10/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 126.6901 - val_loss: 111.8169
Epoch 11/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 122.1317 - val_

331

In [38]:
# ==================== 10. MODEL 4: Mutual mRMR ====================
K_MRMR = 10
X_train_mrmr, X_val_mrmr, X_test_mrmr = apply_mrmr(
    X_train_flat, y_train, X_val_flat, X_test_flat, k=K_MRMR
)
print(f"  mRMR: Selected top {K_MRMR} features")

model_mrmr, hist_mrmr = build_and_train_lstm(X_train_mrmr, y_train, X_val_mrmr, y_val,
                                              "mRMR", X_train_mrmr.shape[1])
results_mRMR = evaluate_model(model_mrmr, X_test_mrmr, y_test, "mRMR")

print(f"mRMR RMSE: {results_mRMR['RMSE']:.2f} mg/dL")
print(f"mRMR MAE: {results_mRMR['MAE']:.2f} mg/dL")
print(f"mRMR Accuracy: {results_mRMR['Accuracy (%)']:.2f}%")
print(f"mRMR MAPE: {results_mRMR['MAPE (%)']:.2f}%")
print(f"mRMR MAPE Accuracy: {results_mRMR['MAPE Accuracy (%)']:.2f}%")


del X_train_mrmr, X_val_mrmr, X_test_mrmr, model_mrmr
gc.collect()

  mRMR: Selected top 10 features

  Training mRMR
Epoch 1/16


C:\Users\Roaa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1799/1799 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step - loss: 17301.4336 - val_loss: 6396.6875
Epoch 2/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 5194.7993 - val_loss: 1594.7036
Epoch 3/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 1791.9197 - val_loss: 542.7362
Epoch 4/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 713.8337 - val_loss: 246.9424
Epoch 5/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 329.5268 - val_loss: 159.9656
Epoch 6/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 205.8835 - val_loss: 138.2891
Epoch 7/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 159.9356 - val_loss: 123.8324
Epoch 8/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 141.3814 - val_loss: 121.0472
Epoch 9/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 134.9082 - val_loss: 117.5055
Epoch 10/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 131.7652 - val_loss: 113.7482
Epoch 11/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 128.2922 -

331

In [39]:
# ==================== 11. MODEL 5: PCA ====================
N_COMPONENTS = 10
X_train_pca, X_val_pca, X_test_pca = apply_pca(
    X_train_flat, X_val_flat, X_test_flat, n_components=N_COMPONENTS
)
print(f"  PCA: Reduced to {N_COMPONENTS} components")

model_pca, hist_pca = build_and_train_lstm(X_train_pca, y_train, X_val_pca, y_val,
                                            "PCA", X_train_pca.shape[1])
results_pca = evaluate_model(model_pca, X_test_pca, y_test, "PCA")

print(f"PCA RMSE: {results_pca['RMSE']:.2f} mg/dL")
print(f"PCA MAE: {results_pca['MAE']:.2f} mg/dL")
print(f"PCA Accuracy: {results_pca['Accuracy (%)']:.2f}%")
print(f"PCA MAPE: {results_pca['MAPE (%)']:.2f}%")
print(f"PCA MAPE Accuracy: {results_pca['MAPE Accuracy (%)']:.2f}%")

del X_train_pca, X_val_pca, X_test_pca, model_pca
gc.collect()

  PCA variance: 99.18%
  PCA: Reduced to 10 components

  Training PCA
Epoch 1/16


C:\Users\Roaa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1799/1799 ━━━━━━━━━━━━━━━━━━━━ 13s 6ms/step - loss: 15384.7734 - val_loss: 5139.2783
Epoch 2/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 4889.1216 - val_loss: 1860.5364
Epoch 3/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 1928.4695 - val_loss: 718.7255
Epoch 4/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 870.3621 - val_loss: 471.7905
Epoch 5/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 533.3660 - val_loss: 346.3317
Epoch 6/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 400.5500 - val_loss: 310.3630
Epoch 7/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 357.6019 - val_loss: 326.2661
Epoch 8/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 330.3736 - val_loss: 294.9334
Epoch 9/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 315.1719 - val_loss: 284.8466
Epoch 10/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 309.4465 - val_loss: 266.1060
Epoch 11/16
1799/1799 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 293.3538 -

331

In [41]:
# ==================== 12. FINAL COMPARISON ====================
# Compile all results
all_results = [
    baseline_results,
    results_raw,
    results_manual,
    results_fisher,
    results_mRMR,
    results_pca
]

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('RMSE')

print("\n" + results_df.to_string(index=False))

# Find best model
best_model = results_df.iloc[0]['Model']
best_rmse = results_df.iloc[0]['RMSE']
best_acc = results_df.iloc[0]['Accuracy (%)']

print(f"BEST MODEL: {best_model}")
print(f"RMSE: {best_rmse:.2f} mg/dL")
print(f"Accuracy: {best_acc:.2f}%")


           Model        MSE      RMSE       MAE  Accuracy (%)  MAPE (%)  MAPE Accuracy (%)
    Fisher Score 121.043976 11.001999  6.617703     90.666937  4.436355          95.563644
    Raw Features 121.332077 11.015084  6.567020     90.934523  4.387756          95.612244
            mRMR 127.592400 11.295681  6.728521     90.160146  4.475711          95.524292
Baseline (Naive) 217.547867 14.749504  9.720819     80.904115  6.409017          93.590981
 Manual Features 257.912079 16.059641 10.824919     76.578147  7.067797          92.932205
             PCA 271.257599 16.469900 11.350388     74.194202  7.803717          92.196281
BEST MODEL: Fisher Score
RMSE: 11.00 mg/dL
Accuracy: 90.67%


In [42]:
# ==================== 13. VISUALIZATIONS ====================

# Plot 1: RMSE Comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# RMSE Bar Chart
axes[0].bar(results_df['Model'], results_df['RMSE'], color='steelblue')
axes[0].set_xlabel('Model', fontsize=12)
axes[0].set_ylabel('RMSE (mg/dL)', fontsize=12)
axes[0].set_title('RMSE Comparison', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Accuracy Bar Chart
axes[1].bar(results_df['Model'], results_df['Accuracy (%)'], color='coral')
axes[1].set_xlabel('Model', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Accuracy Comparison (±15 mg/dL)', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

# MAE Bar Chart
axes[2].bar(results_df['Model'], results_df['MAE'], color='seagreen')
axes[2].set_xlabel('Model', fontsize=12)
axes[2].set_ylabel('MAE (mg/dL)', fontsize=12)
axes[2].set_title('MAE Comparison', fontsize=14, fontweight='bold')
axes[2].tick_params(axis='x', rotation=45)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison_lstm.png', dpi=300, bbox_inches='tight')
print("Saved: model_comparison_lstm.png")
plt.close()

# Plot 2: Training History
fig2, axes2 = plt.subplots(2, 3, figsize=(18, 10))
axes2 = axes2.flatten()

histories = [
    (hist_raw, 'Raw Features'),
    (hist_manual, 'Manual Features'),
    (hist_fisher, 'Fisher Score'),
    (hist_mrmr, 'mRMR'),
    (hist_pca, 'PCA')
]

for idx, (hist, name) in enumerate(histories):
    axes2[idx].plot(hist.history['loss'], label='Train Loss', linewidth=2)
    axes2[idx].plot(hist.history['val_loss'], label='Val Loss', linewidth=2)
    axes2[idx].set_xlabel('Epoch', fontsize=10)
    axes2[idx].set_ylabel('Loss (MSE)', fontsize=10)
    axes2[idx].set_title(f'{name}', fontsize=12, fontweight='bold')
    axes2[idx].legend()
    axes2[idx].grid(True, alpha=0.3)

axes2[5].axis('off')

plt.tight_layout()
plt.savefig('training_histories_lstm.png', dpi=300, bbox_inches='tight')
print("Saved: training_histories_lstm.png")
plt.close()

Saved: model_comparison_lstm.png
Saved: training_histories_lstm.png


In [43]:
# ==================== 14. SAVE RESULTS ====================
results_df.to_csv('model_comparison_results_lstm.csv', index=False)
print("Saved: model_comparison_results_lstm.csv")

# Final cleanup
del X_train_flat, X_val_flat, X_test_flat, y_train, y_val, y_test
gc.collect()

Saved: model_comparison_results_lstm.csv


4894

In [ ]:
# ==================== HYPERPARAMETER TUNING ====================

X_all = []
y_all = []
for b in range(NUM_BATCHES):
    X_all.append(np.load(os.path.join(SAVE_PATH, f"X_batch_{b}.npy")))
    y_all.append(np.load(os.path.join(SAVE_PATH, f"y_batch_{b}.npy")))
X_all = np.concatenate(X_all, axis=0)
y_all = np.concatenate(y_all, axis=0)

# Split
n = len(X_all)
train_size = int(0.70 * n)
val_size = int(0.15 * n)

X_train_flat = X_all[:train_size]
y_train = y_all[:train_size].flatten()
X_val_flat = X_all[train_size:train_size + val_size]
y_val = y_all[train_size:train_size + val_size].flatten()
X_test_flat = X_all[train_size + val_size:]
y_test = y_all[train_size + val_size:].flatten()

del X_all, y_all
gc.collect()

# Apply Fisher Score
X_train_tune, X_val_tune, X_test_tune = apply_fisher_score(
    X_train_flat, y_train, X_val_flat, X_test_flat, k=10
)

print(f"Data ready: Train={len(X_train_tune)}, Val={len(X_val_tune)}, Test={len(X_test_tune)}")

# Configs
configs = [
    {'lstm_units': 64, 'dropout': 0.0, 'lr': 0.001, 'batch_size': 64, 'layers': 1},
    {'lstm_units': 128, 'dropout': 0.2, 'lr': 0.001, 'batch_size': 64, 'layers': 1},
    {'lstm_units': 64, 'dropout': 0.2, 'lr': 0.001, 'batch_size': 64, 'layers': 2},
    {'lstm_units': 64, 'dropout': 0.2, 'lr': 0.0001, 'batch_size': 64, 'layers': 1},
    {'lstm_units': 128, 'dropout': 0.3, 'lr': 0.0005, 'batch_size': 64, 'layers': 2},
]

# Test configs
results = []
for i, cfg in enumerate(configs):
    print(f"\nConfig {i+1}/{len(configs)}: Units={cfg['lstm_units']}, Dropout={cfg['dropout']}, LR={cfg['lr']}, Layers={cfg['layers']}")
    
    tf.keras.backend.clear_session()
    gc.collect()
    
    X_tr = X_train_tune[..., np.newaxis]
    X_v = X_val_tune[..., np.newaxis]
    X_te = X_test_tune[..., np.newaxis]
    
    from tensorflow.keras.layers import Dropout
    model = Sequential()
    
    if cfg['layers'] == 1:
        model.add(LSTM(cfg['lstm_units'], input_shape=(X_tr.shape[1], 1)))
    else:
        model.add(LSTM(cfg['lstm_units'], return_sequences=True, input_shape=(X_tr.shape[1], 1)))
        if cfg['dropout'] > 0:
            model.add(Dropout(cfg['dropout']))
        model.add(LSTM(cfg['lstm_units']))
    
    if cfg['dropout'] > 0:
        model.add(Dropout(cfg['dropout']))
    model.add(Dense(1))
    
    model.compile(optimizer=Adam(learning_rate=cfg['lr']), loss='mse')
    
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)
    model.fit(X_tr, y_train, validation_data=(X_v, y_val), epochs=20, 
              batch_size=cfg['batch_size'], callbacks=[early_stop], verbose=0, shuffle=True)
    
    y_pred = model.predict(X_te, verbose=0, batch_size=64).reshape(-1)
    errors = y_test - y_pred
    
    mse = np.mean(errors ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(errors))
    acc = (np.sum(np.abs(errors) <= 15) / len(y_test)) * 100
    mape = np.mean(np.abs(errors) / np.abs(y_test)) * 100
    mape_acc = 100 - mape
    
    print(f"  RMSE: {rmse:.2f}, MAE: {mae:.2f}, Acc: {acc:.2f}%, MAPE: {mape:.2f}%")
    
    results.append({'Config': i+1, 'MSE': mse, 'RMSE': rmse, 'MAE': mae, 
                   'Accuracy': acc, 'MAPE': mape, 'MAPE_Accuracy': mape_acc, **cfg})
    
    del model, X_tr, X_v, X_te, y_pred
    gc.collect()

# Results
results_df = pd.DataFrame(results).sort_values('RMSE')
print(results_df.to_string(index=False))

best = results_df.iloc[0]
print(f"\n BEST: Config {best['Config']} - RMSE: {best['RMSE']:.2f}, MAE: {best['MAE']:.2f}, Acc: {best['Accuracy']:.2f}%, MAPE: {best['MAPE']:.2f}%")

results_df.to_csv('tuning_results.csv', index=False)
print(" Saved: tuning_results.csv")

del X_train_flat, X_val_flat, X_test_flat
del X_train_tune, X_val_tune, X_test_tune
del y_train, y_val, y_test
gc.collect()

Data ready: Train=115096, Val=24663, Test=24665

Config 1/5: Units=64, Dropout=0.0, LR=0.001, Layers=1


C:\Users\Roaa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  RMSE: 10.99, MAE: 6.55, Acc: 90.88%, MAPE: 4.42%

Config 2/5: Units=128, Dropout=0.2, LR=0.001, Layers=1
  RMSE: 10.92, MAE: 6.49, Acc: 90.96%, MAPE: 4.32%

Config 3/5: Units=64, Dropout=0.2, LR=0.001, Layers=2
  RMSE: 11.09, MAE: 6.61, Acc: 90.40%, MAPE: 4.37%

Config 4/5: Units=64, Dropout=0.2, LR=0.0001, Layers=1
  RMSE: 41.40, MAE: 22.23, Acc: 67.62%, MAPE: 10.13%

Config 5/5: Units=128, Dropout=0.3, LR=0.0005, Layers=2
  RMSE: 11.07, MAE: 6.51, Acc: 90.55%, MAPE: 4.26%
 Config         MSE      RMSE       MAE  Accuracy      MAPE  MAPE_Accuracy  lstm_units  dropout     lr  batch_size  layers
      2  119.276039 10.921357  6.487757 90.958849  4.324127      95.675873         128      0.2 0.0010          64       1
      1  120.732109 10.987817  6.552817 90.877762  4.415878      95.584122          64      0.0 0.0010          64       1
      5  122.581352 11.071647  6.508310 90.545307  4.262773      95.737228         128      0.3 0.0005          64       2
      3  122.959747 11.0887

0